# Wellness Centre ERPNext Migration**Version:** 1.0  **Created:** March 2026  **ERPNext:** v15  **Site:** well.rosslyn.cloud---## Migration OverviewComplete historical data migration from 18 CSV files into ERPNext.**Data Scope:**- 220 Sales Invoices (KES 2,589,840)- 220 Payment Entries - 709 Expense transactions (KES 4,363,477)- 3 Capital injections (KES 4,000,000)- 15 Savings transfers (KES 229,000)- Master data (customers, suppliers, items)**Migration Phases:**- Phase 0: Prerequisites & Setup- Phase 1: Sales Invoices & Payments- Phase 2: Financial Transactions (Expenses, Capital, Savings)- Phase 3-6: TBD (Inventory, Operations, Compliance)---

# Phase 0: Environment SetupInitialize Python environment, connect to ERPNext, load data, validate prerequisites.**Estimated Time:** 5 minutes

In [2]:
# Standard library
from pathlib import Path
from datetime import datetime
import csv
import sys

# Third party
import pandas as pd
from frappeclient import FrappeClient

# Add src to path for imports
REPO_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
SRC_DIR = REPO_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

# Paths
DATA_DIR = Path(REPO_ROOT,'data/wellness_centre')  # CSV files
OUTPUTS_DIR = REPO_ROOT / 'user-data' / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Repo root: {REPO_ROOT}")
print(f"✓ Source: {SRC_DIR}")
print(f"✓ Data: {DATA_DIR}")
print(f"✓ Outputs: {OUTPUTS_DIR}")

✓ Repo root: /home/jovyan/work/ERP/emt
✓ Source: /home/jovyan/work/ERP/emt/src
✓ Data: /home/jovyan/work/ERP/emt/data/wellness_centre
✓ Outputs: /home/jovyan/work/ERP/emt/user-data/outputs


In [3]:
# Load connection config
import yaml
import csv
from pathlib import Path

CONFIG_FILE = REPO_ROOT / 'config' / 'erpnext_connection.yaml'

if not CONFIG_FILE.exists():
    raise FileNotFoundError(
        f"Config not found: {CONFIG_FILE}\n"
        f"Copy config/erpnext_connection.yaml.example and fill in credentials"
    )

with open(CONFIG_FILE) as f:
    config = yaml.safe_load(f)

ERPNEXT_URL = config['url']

# Option 1: Try CSV file first (if specified)
if 'api_csv' in config and config['api_csv']:
    csv_path = REPO_ROOT / config['api_csv']
    
    if csv_path.exists():
        print(f"✓ Loading API keys from CSV: {csv_path.name}")
        with open(csv_path) as f:
            reader = csv.DictReader(f)
            row = next(reader)
            API_KEY = row['api_key']
            API_SECRET = row['api_secret']
        print("  Source: CSV file")
    else:
        print(f"⚠ CSV file not found: {csv_path}")
        print("  Falling back to YAML credentials...")
        API_KEY = config['api_key']
        API_SECRET = config['api_secret']
        print("  Source: YAML file")
else:
    # Option 2: Use YAML credentials
    API_KEY = config['api_key']
    API_SECRET = config['api_secret']
    print("  Source: YAML file")

print(f"✓ Config loaded")
print(f"  URL: {ERPNEXT_URL}")

✓ Loading API keys from CSV: api_keys.csv
  Source: CSV file
✓ Config loaded
  URL: http://erpnext-frontend:8080


In [4]:
# Connect to ERPNext
client = FrappeClient(ERPNEXT_URL)
client.authenticate(API_KEY, API_SECRET)

# Add Host header if using internal Docker URL (for DNS multitenant)
if 'erpnext-frontend' in ERPNEXT_URL or 'localhost' in ERPNEXT_URL:
    # Get domain from config or default
    domain = config.get('headers', {}).get('Host', 'well.rosslyn.cloud')
    client.session.headers.update({"Host": domain})
    print(f"✓ Added Host header: {domain}")

# Test connection
try:
    company = client.get_doc("Company", "Wellness Centre")
    print("=" * 70)
    print("ERPNEXT CONNECTION SUCCESSFUL")
    print("=" * 70)
    print(f"Company: {company.get('company_name')}")
    print(f"Currency: {company.get('default_currency')}")
    print(f"Country: {company.get('country')}")
    print("=" * 70)
except Exception as e:
    print(f"✗ Connection failed: {e}")
    raise

✓ Added Host header: well.rosslyn.cloud
ERPNEXT CONNECTION SUCCESSFUL
Company: Wellness Centre
Currency: KES
Country: Kenya


In [6]:
# Load all source CSV files
print("LOADING SOURCE DATA")
print("=" * 70)

# Transactions and categories
transactions_df = pd.read_csv(DATA_DIR / 'transactions.csv')
categories_df = pd.read_csv(DATA_DIR / 'transaction_categories.csv')

# Merge for category names
tx = transactions_df.merge(
    categories_df,
    left_on='category_id',
    right_on='id',
    suffixes=('', '_cat')
)

# Invoice data
invoices_df = pd.read_csv(DATA_DIR / 'etims_invoices.csv')
items_df = pd.read_csv(DATA_DIR / 'etims_invoice_items.csv')

# Contact data
contacts_df = pd.read_csv(DATA_DIR / 'contacts.csv')
contact_types_df = pd.read_csv(DATA_DIR / 'contact_types.csv')

print(f"✓ Transactions:     {len(transactions_df):,}")
print(f"✓ Invoices:         {len(invoices_df):,}")
print(f"✓ Invoice Items:    {len(items_df):,}")
print(f"✓ Contacts:         {len(contacts_df):,}")
print("=" * 70)

LOADING SOURCE DATA
✓ Transactions:     947
✓ Invoices:         220
✓ Invoice Items:    220
✓ Contacts:         45


## Prerequisites ValidationVerify ERPNext is properly configured before importing data.

In [8]:
# Check and auto-fix prerequisites
from setup.prerequisites_checker import PrerequisitesChecker

checker = PrerequisitesChecker(
    client, 
    company="Wellness Centre", 
    data_dir=DATA_DIR
)

# No transactions_df parameter needed - scans CSV files directly
status = checker.check_and_fix_all()

print(status['report'])

if not status['ready']:
    raise SystemExit("❌ CRITICAL: Fix issues above before continuing")

PREREQUISITES CHECK

COMPREHENSIVE DATE SCAN:
  Overall range: 2024-01-05 to 2025-02-28
  Files scanned: 13

  Date ranges by file:
    transactions.csv               transaction_date    : 2024-01-05 to 2025-02-11 (947 records)
    etims_invoices.csv             invoice_date        : 2024-03-01 to 2025-02-11 (220 records)
    etims_invoices.csv             transmission_date   : 2024-03-01 to 2025-02-11 (220 records)
    room_bookings.csv              check_in_date       : 2024-03-16 to 2025-02-11 (54 records)
    room_bookings.csv              check_out_date      : 2024-03-17 to 2025-02-13 (54 records)
    events.csv                     event_date          : 2024-03-16 to 2025-01-11 (25 records)
    events.csv                     end_date            : 2024-03-16 to 2025-01-12 (25 records)
    egg_sales.csv                  sale_date           : 2024-03-01 to 2025-02-07 (103 records)
    egg_production.csv             week_start_date     : 2024-02-12 to 2025-02-03 (52 records)
    egg_p

## Custom Fields Creation**CRITICAL:** Create custom fields for duplicate detection BEFORE any imports.These fields persist in ERPNext even after snapshot restores.

In [28]:
# Create custom field for Journal Entry duplicate detection
print("=" * 70)
print("CREATING CUSTOM FIELD: Journal Entry - source_transaction_id")
print("=" * 70)

custom_field = {
    "doctype": "Custom Field",
    "dt": "Journal Entry",
    "fieldname": "source_transaction_id",
    "label": "Source Transaction ID",
    "fieldtype": "Data",
    "insert_after": "title",
    "read_only": 1,
    "allow_on_submit": 1,
    "in_list_view": 0,
    "in_standard_filter": 1
}

try:
    # Check if already exists
    existing = client.get_list(
        "Custom Field",
        filters={
            "dt": "Journal Entry",
            "fieldname": "source_transaction_id"
        }
    )
    
    if existing:
        print("✓ Custom field already exists")
    else:
        client.insert(custom_field)
        print("✓ Created custom field: Journal Entry-source_transaction_id")
except Exception as e:
    print(f"✗ Error: {e}")

print("=" * 70)

CREATING CUSTOM FIELD: Journal Entry - source_transaction_id
✓ Created custom field: Journal Entry-source_transaction_id


In [ ]:
# Create custom field for Sales Invoice duplicate detection
print("Creating custom field: original_invoice_number on Sales Invoice")
print("=" * 70)

try:
    si_field = client.insert({
        "doctype": "Custom Field",
        "dt": "Sales Invoice",
        "fieldname": "original_invoice_number",
        "label": "Original Invoice Number",
        "fieldtype": "Data",
        "insert_after": "naming_series",
        "read_only": 1,
        "no_copy": 1,
        "unique": 1,
        "description": "Original invoice number from source system"
    })
    print("✓ Created")
except Exception as e:
    if "already exists" in str(e).lower():
        print("✓ Already exists")
    else:
        print(f"✗ Error: {e}")

print("=" * 70)

## Master Data CreationCreate customers, suppliers, and items from CSV data.

In [9]:
# Auto-create all master data
from setup.master_data_creator import MasterDataCreator

creator = MasterDataCreator(client, "Wellness Centre")
results = creator.create_all(
    contacts_df=contacts_df,
    contact_types_df=contact_types_df,
    invoices_df=invoices_df,
    invoice_items_df=items_df
)

creator.print_summary()

CREATING MASTER DATA
Creating UOMs...
  Created: 0, Existed: 4, Errors: 0
Creating Customers...
  Created: 0, Existed: 13, Errors: 0
Creating Suppliers...
  Created: 0, Existed: 9, Errors: 0
Creating Service Items...
  Created: 0, Existed: 10, Errors: 0

MASTER DATA CREATION SUMMARY

UOMS:
  Created: 0
  Existed: 4
  Errors:  0

CUSTOMERS:
  Created: 0
  Existed: 13
  Errors:  0

SUPPLIERS:
  Created: 0
  Existed: 9
  Errors:  0

ITEMS:
  Created: 0
  Existed: 10
  Errors:  0


## Notification HelperAudio notification for long-running imports.

In [12]:
# Notification sound cell - run this first to define the function
from IPython.display import Audio, display
import numpy as np

def notify(sound_type='success'):
    """Play notification sound. Types: 'success', 'error', 'complete'"""
    sample_rate = 22050
    duration = 0.3
    
    if sound_type == 'success':
        # Pleasant two-tone chime
        freq1, freq2 = 523, 659  # C5, E5
    elif sound_type == 'error':
        # Lower warning tone
        freq1, freq2 = 200, 180
    else:  # complete
        # Three-tone ascending
        freq1, freq2 = 440, 554  # A4, C#5
    
    # Generate tones
    t = np.linspace(0, duration, int(sample_rate * duration))
    tone1 = np.sin(2 * np.pi * freq1 * t[:len(t)//2])
    tone2 = np.sin(2 * np.pi * freq2 * t[len(t)//2:])
    sound = np.concatenate([tone1, tone2])
    
    # Fade in/out to avoid clicks
    fade = np.linspace(0, 1, len(sound)//10)
    sound[:len(fade)] *= fade
    sound[-len(fade):] *= fade[::-1]
    
    display(Audio(sound, rate=sample_rate, autoplay=True))

# Test it
notify('success')

---## ✓ Phase 0 CompleteEnvironment ready for data import.---

# Phase 1: Sales Invoices & Payment EntriesImport all sales transactions and payments.**Expected Results:**- 220 Sales Invoices imported- 220 Payment Entries created- Zero outstanding receivables**Estimated Time:** 15 minutes

## Phase 1A: Sales Invoices

In [ ]:
# Reload Phase 1A modules
import importlib
from orchestration import sales_invoice_importer

importlib.reload(sales_invoice_importer)
from orchestration.sales_invoice_importer import SalesInvoiceImporter

print("✓ Phase 1A modules reloaded")

In [13]:
# PHASE 1A: Import Sales Invoices
print("=" * 70)
print("PHASE 1A: IMPORTING SALES INVOICES")
print("=" * 70)

from orchestration.sales_invoice_importer import SalesInvoiceImporter

# Use v3.0 sequential importer (proven reliable, optimal for API imports)
importer = SalesInvoiceImporter(
    client,
    "Wellness Centre"
)

# Import batch with timing metrics
results = importer.import_batch(
    invoices_df=invoices_df,
    invoice_items_df=items_df,
    contacts_df=contacts_df  # Required by v3.0 signature (even if unused internally)
)

# Display summary with performance metrics
print(importer.get_summary())

PHASE 1A: IMPORTING SALES INVOICES
[SalesInvoiceImporter 3.0-duplicate-prevention]
Importing 220 sales invoices...
  Skipped 50 duplicates...
  Skipped 100 duplicates...
  Skipped 150 duplicates...
  Skipped 200 duplicates...
SALES INVOICE IMPORT SUMMARY
Successful: 0
Skipped:    220 (already exist)
Failed:     0

Performance:
  Duration: 46.83 seconds (0.78 minutes)
  Rate: 0.0 invoices/second


## Account Creation Policy Configuration

**IMPORTANT:** Control how accounts are created during migration.

**Three Modes Available:**
1. **AUTOMATIC** (Recommended for initial migration)  
   - Creates all accounts without prompting  
   - Fast, efficient for trusted configurations  
   - Default mode if not specified  

2. **CONFIRM** (Interactive review)  
   - Prompts for confirmation before creating each account  
   - Good for cautious users or production systems  
   - Allows selective account creation  

3. **MANUAL** (No automatic creation)  
   - Raises error if account doesn't exist  
   - User must create accounts manually in ERPNext  
   - Maximum control, slower process  

**Fine-Grained Control:**  
You can also set different policies for different account types (e.g., manual for Equity, automatic for Expenses).

**For this migration:** We use AUTOMATIC mode for efficient initial data load.


In [5]:
# Diagnostic: Check Python path and module structure
import sys
from pathlib import Path

print("Python sys.path:")
for p in sys.path[:5]:
    print(f"  {p}")

print("\nChecking paths:")
repo_root = Path.home() / "work/ERP/emt"
src_dir = repo_root / "src"
core_dir = src_dir / "core"
policy_file = core_dir / "account_creation_policy.py"
init_file = core_dir / "__init__.py"

print(f"  SRC_DIR exists: {src_dir.exists()}")
print(f"  CORE_DIR exists: {core_dir.exists()}")
print(f"  __init__.py exists: {init_file.exists()}")
print(f"  account_creation_policy.py exists: {policy_file.exists()}")

print(f"\nSRC_DIR in sys.path: {str(src_dir) in sys.path}")

# Try direct import
if str(src_dir) in sys.path:
    try:
        import core.account_creation_policy
        print("\n✓ Import successful!")
    except Exception as e:
        print(f"\n✗ Import failed: {e}")
else:
    print("\n✗ SRC_DIR not in sys.path - this is the problem!")

Python sys.path:
  /opt/conda/lib/python311.zip
  /opt/conda/lib/python3.11
  /opt/conda/lib/python3.11/lib-dynload
  
  /home/jovyan/venvs/erpnext/lib/python3.11/site-packages

Checking paths:
  SRC_DIR exists: True
  CORE_DIR exists: True
  __init__.py exists: True
  account_creation_policy.py exists: True

SRC_DIR in sys.path: False

✗ SRC_DIR not in sys.path - this is the problem!


In [4]:
# Configure Account Creation Policy
from core.account_creation_policy import AccountCreationPolicy

# AUTOMATIC mode: Create all accounts without confirmation (recommended for migration)
policy = AccountCreationPolicy(mode=AccountCreationPolicy.AUTOMATIC)

# Alternative modes (uncomment to use):
# ------------------------------------

# CONFIRM mode: Prompt before each account creation
# policy = AccountCreationPolicy(mode=AccountCreationPolicy.CONFIRM)

# MANUAL mode: No auto-creation, raise error if account missing
# policy = AccountCreationPolicy(mode=AccountCreationPolicy.MANUAL)

# MIXED mode: Different policies for different account types
# policy = AccountCreationPolicy(
#     mode=AccountCreationPolicy.AUTOMATIC,
#     overrides={
#         "Equity": AccountCreationPolicy.CONFIRM,  # Confirm equity accounts
#         "Expense": AccountCreationPolicy.AUTOMATIC  # Auto-create expenses
#     }
# )

print(f"✓ Account Creation Policy: {policy.mode}")
if policy.overrides:
    print(f"  Overrides: {policy.overrides}")

ModuleNotFoundError: No module named 'core'

## Initialize AccountRegistryThe AccountRegistry dynamically discovers and creates accounts as needed.It now integrates with the AccountCreationPolicy configured above to control account creation behavior.

In [ ]:
# Initialize AccountRegistry with Account Creation Policy
from orchestration.account_registry import AccountRegistry

# Create registry with configured policy
registry = AccountRegistry(
    client=client,
    company="Wellness Centre",
    policy=policy  # Uses policy configured in previous cell
)

print(f"✓ AccountRegistry initialized")
print(f"  Company: Wellness Centre")
print(f"  Suffix: {registry.suffix}")
print(f"  Policy: {policy.mode}")

## Phase 1B: Payment Entries

In [17]:
# Phase 1B: Import Payment Entries
from orchestration.payment_entry_importer import PaymentEntryImporter

print("=" * 70)
print("PHASE 1B: IMPORTING PAYMENT ENTRIES")
print("=" * 70)

# Initialize importer with AccountRegistry
payment_imp = PaymentEntryImporter(client, "Wellness Centre", registry)

# Import payments
results = payment_imp.import_batch(transactions_df, invoices_df)

# Show results
print("=" * 70)
print("PAYMENT ENTRY IMPORT SUMMARY")
print("=" * 70)
print(f"Successful: {results['successful']}")
print(f"Skipped:    {results['skipped']} (already exist)")
print(f"Failed:     {results['failed']}")
print()
print("Performance:")
print(f"  Duration: {results['duration_seconds']} seconds ({results['duration_seconds']/60:.2f} minutes)")
print(f"  Rate: {results['rate_per_second']} payments/second")
print("=" * 70)

notify('complete')

PHASE 1B: IMPORTING PAYMENT ENTRIES
[PaymentEntryImporter 3.2-accountregistry]
Importing 220 payment entries...
  Skipped 50 (already paid)...
  Skipped 100 (already paid)...
  Skipped 150 (already paid)...
  Skipped 200 (already paid)...
PAYMENT ENTRY IMPORT SUMMARY
Successful: 0
Skipped:    220 (already exist)
Failed:     0

Performance:
  Duration: 41.6 seconds (0.69 minutes)
  Rate: 0.0 payments/second


---## ✓ Phase 1 CompleteSuccessfully imported 220 invoices and 220 payments.**Next Step:** Create snapshot before Phase 2```bash/opt/consultancy-platform/scripts/erpnext/erpnext-site-snapshot \    well.rosslyn.cloud create "Phase 1 Complete - Invoices & Payments"```---

# Phase 2: Financial TransactionsImport all remaining financial transactions as Journal Entries.**Scope:**- 709 Expense transactions (KES 4,363,477)- 3 Capital injections (KES 4,000,000)- 15 Savings transfers (KES 229,000)**Total:** 727 Journal Entries, KES 8,592,477**Estimated Time:** 15 minutes

## Reload Updated ModulesEnsure latest importer code is loaded.

In [ ]:
# Reload Phase 2 modules
import importlib
from orchestration import (
    account_registry,
    account_mapper,
    expense_importer,
    capital_injection_importer,
    savings_transfer_importer
)

importlib.reload(account_registry)
importlib.reload(account_mapper)
importlib.reload(expense_importer)
importlib.reload(capital_injection_importer)
importlib.reload(savings_transfer_importer)

from orchestration.account_mapper import AccountMapper
from orchestration.expense_importer import ExpenseImporter
from orchestration.capital_injection_importer import CapitalInjectionImporter
from orchestration.savings_transfer_importer import SavingsTransferImporter

print("✓ Phase 2 modules reloaded")
print(f"  ExpenseImporter: {ExpenseImporter.VERSION}")
print(f"  CapitalInjectionImporter: {CapitalInjectionImporter.VERSION}")
print(f"  SavingsTransferImporter: {SavingsTransferImporter.VERSION}")

## Phase 2A: Expense TransactionsImport 709 expenses with automatic account creation.

In [69]:
# ============================================================================
# PHASE 2A: IMPORT EXPENSES
# ============================================================================
print("=" * 70)
print("PHASE 2A: IMPORTING EXPENSES")
print("=" * 70)

# Step 1: Load transaction categories and prepare account mappings
from pathlib import Path
import pandas as pd
from orchestration.account_mapper import AccountMapper

# Load categories from CSV
categories_df = pd.read_csv(DATA_DIR / 'transaction_categories.csv')
print(f"Loaded {len(categories_df)} transaction categories")

# Initialize mapper with YAML configuration
config_file = Path.home() / "work/ERP/emt/config/account_mappings.yaml"
account_mapper = AccountMapper(
    config_file=config_file,
    company="Wellness Centre"
)

# Map expense categories to accounts
account_mappings = account_mapper.map_categories(
    categories_df,
    transaction_type='expense'
)
print(f"Mapped {len(account_mappings)} expense categories to accounts")

# Create missing accounts in ERPNext
account_results = account_mapper.create_missing_accounts(client, account_mappings)
print(f"  Created: {len(account_results['created'])} new accounts")
print(f"  Already existed: {len(account_results['existed'])} accounts")
if account_results['errors']:
    print(f"  ⚠ Errors: {len(account_results['errors'])}")
    for error in account_results['errors']:
        print(f"    - {error['category']}: {error['error']}")
print()

# Step 2: Import expenses using the mappings
expense_imp = ExpenseImporter(client, "Wellness Centre", registry)
expense_results = expense_imp.import_expenses(
    transactions_df,
    account_mappings
)

expense_imp.print_summary()

notify('success')

PHASE 2A: IMPORTING EXPENSES
Loaded 31 transaction categories
Mapped 24 expense categories to accounts
  Created: 17 new accounts
  Already existed: 0 accounts

Importing 709 expense transactions...
  Progress: 50/709 (✓ 50, ⊘ 0, ✗ 0)
  Progress: 100/709 (✓ 100, ⊘ 0, ✗ 0)
  Progress: 150/709 (✓ 150, ⊘ 0, ✗ 0)
  Progress: 200/709 (✓ 200, ⊘ 0, ✗ 0)
  Progress: 250/709 (✓ 250, ⊘ 0, ✗ 0)
  Progress: 300/709 (✓ 300, ⊘ 0, ✗ 0)
  Progress: 350/709 (✓ 350, ⊘ 0, ✗ 0)
  Progress: 400/709 (✓ 400, ⊘ 0, ✗ 0)
  Progress: 450/709 (✓ 450, ⊘ 0, ✗ 0)
  Progress: 500/709 (✓ 500, ⊘ 0, ✗ 0)
  Progress: 550/709 (✓ 550, ⊘ 0, ✗ 0)
  Progress: 600/709 (✓ 600, ⊘ 0, ✗ 0)
  Progress: 650/709 (✓ 650, ⊘ 0, ✗ 0)
  Progress: 700/709 (✓ 700, ⊘ 0, ✗ 0)
  Progress: 709/709 (✓ 709, ⊘ 0, ✗ 0)

EXPENSE IMPORT SUMMARY
Total attempted:  709
Succeeded:        709
Skipped:          0 (duplicates)
Failed:           0
Success amount:   KES 4,363,477.00


## Phase 2B: Capital InjectionsImport 3 capital injections with equity account auto-creation.

In [70]:
# ============================================================================
# PHASE 2B: IMPORT CAPITAL INJECTIONS
# ============================================================================
print("=" * 70)
print("PHASE 2B: IMPORTING CAPITAL INJECTIONS")
print("=" * 70)

capital_imp = CapitalInjectionImporter(client, "Wellness Centre", registry)
capital_results = capital_imp.import_capital_injections(transactions_df)

capital_imp.print_summary()

notify('success')

PHASE 2B: IMPORTING CAPITAL INJECTIONS
Importing 3 capital injection transactions...
  ✓ 2024-01-15: KES 2,000,000 → ACC-JV-2026-00710
  ✓ 2024-01-22: KES 1,500,000 → ACC-JV-2026-00711
  ✓ 2024-02-20: KES 500,000 → ACC-JV-2026-00712

CAPITAL INJECTION IMPORT SUMMARY
Total attempted:  3
Succeeded:        3
Skipped:          0 (duplicates)
Failed:           0
Success amount:   KES 4,000,000.00
Equity account:   Capital Stock - WC


## Phase 2C: Savings TransfersImport 15 savings transfers with savings account auto-creation.

In [71]:
# ============================================================================
# PHASE 2C: IMPORT SAVINGS TRANSFERS
# ============================================================================
print("=" * 70)
print("PHASE 2C: IMPORTING SAVINGS TRANSFERS")
print("=" * 70)

savings_imp = SavingsTransferImporter(client, "Wellness Centre", registry)
savings_results = savings_imp.import_savings_transfers(transactions_df)

savings_imp.print_summary()

notify('complete')

PHASE 2C: IMPORTING SAVINGS TRANSFERS
  ℹ No savings account found, creating 'Savings Account - WC'
  ✓ Created savings account: Savings Account - WC
Importing 15 savings transfer transactions...
  ✓ 2024-04-26: KES 15,000 → ACC-JV-2026-00713
  ✓ 2024-05-28: KES 10,000 → ACC-JV-2026-00714
  ✓ 2024-06-27: KES 12,000 → ACC-JV-2026-00715
  ✓ 2024-07-25: KES 10,000 → ACC-JV-2026-00716
  ✓ 2024-07-27: KES 15,000 → ACC-JV-2026-00717
  Progress: 5/15 (✓ 5, ⊘ 0, ✗ 0)
  ✓ 2024-08-26: KES 10,000 → ACC-JV-2026-00718
  ✓ 2024-08-27: KES 12,000 → ACC-JV-2026-00719
  ✓ 2024-09-25: KES 12,000 → ACC-JV-2026-00720
  ✓ 2024-09-27: KES 12,000 → ACC-JV-2026-00721
  ✓ 2024-10-25: KES 25,000 → ACC-JV-2026-00722
  Progress: 10/15 (✓ 10, ⊘ 0, ✗ 0)
  ✓ 2024-11-25: KES 20,000 → ACC-JV-2026-00723
  ✓ 2024-11-26: KES 18,000 → ACC-JV-2026-00724
  ✓ 2024-12-25: KES 20,000 → ACC-JV-2026-00725
  ✓ 2024-12-28: KES 18,000 → ACC-JV-2026-00726
  ✓ 2025-01-28: KES 20,000 → ACC-JV-2026-00727
  Progress: 15/15 (✓ 15, ⊘ 0, ✗

## Comprehensive VerificationThree-level verification:1. Quick Summary (counts and totals)2. Detailed Reconciliation (CSV comparison)3. Financial Validation (debits = credits)**Expected Results:**- ✓ 727 Journal Entries- ✓ KES 8,592,477 total- ✓ 0 duplicates- ✓ All entries balanced

In [72]:
# ============================================================================
# VERIFICATION: Detailed Reconciliation
# ============================================================================
from validation.migration_dashboard import MigrationDashboard
import importlib
from validation import migration_dashboard
importlib.reload(migration_dashboard)
from validation.migration_dashboard import MigrationDashboard

dashboard = MigrationDashboard(client, DATA_DIR, "Wellness Centre")

print("\n")
print("=" * 70)
print("QUICK SUMMARY CHECK")
print("=" * 70)
summary = dashboard.quick_summary()

print("\n")
report = dashboard.full_reconciliation()
dashboard.print_reconciliation_report(report)

# Check accounting integrity
print("\n")
integrity = dashboard.validate_accounting_integrity()

# Check outstanding receivables
print("\n")
receivables = dashboard.check_outstanding_receivables()



QUICK SUMMARY CHECK
MIGRATION QUICK SUMMARY
Company: Wellness Centre
Checked: 2026-03-09 05:51:38

Sales Invoices:      220 records, KES 2,589,840
Payment Entries:     220 records, KES 2,589,840
Journal Entries:     727 records, KES 8,592,477

Customers:            13
Suppliers:             9
Items:                10



DETAILED RECONCILIATION REPORT
Company: Wellness Centre
Generated: 2026-03-09T05:51:39.282173

SALES INVOICE:
  CSV Count:           220
  ERPNext Count:       220
  Count Match:        ✓ PASS
  CSV Total:          KES 2,589,840.00
  ERPNext Total:      KES 2,589,840.00
  Amount Match:       ✓ PASS
  Status:             PASS

PAYMENT ENTRY:
  CSV Count:           220
  ERPNext Count:       220
  Count Match:        ✓ PASS
  CSV Total:          KES 2,589,840.00
  ERPNext Total:      KES 2,589,840.00
  Amount Match:       ✓ PASS
  Status:             PASS

JOURNAL ENTRY:
  CSV Count:           727
  ERPNext Count:       727
  Count Match:        ✓ PASS
  CSV Total:     

---## ✓ Phase 2 CompleteSuccessfully imported all financial transactions.**Total Migration Progress:**- Phase 0: ✅ Complete- Phase 1: ✅ Complete (440 transactions)- Phase 2: ✅ Complete (727 transactions)- **Total:** 1,167 transactions, KES 11,182,157**Next Steps:**1. Create Phase 2 snapshot:```bash/opt/consultancy-platform/scripts/erpnext/erpnext-site-snapshot \    well.rosslyn.cloud create "Phase 2 Complete - All Financial Transactions"```2. Git commit:```bashcd ~/work/ERP/emtgit add -Agit commit -m "Phase 2 Complete: All financial transactions migrated"git tag -a v1.3-phase2-complete -m "Phase 1 & 2: 1,167 transactions"```3. Proceed to Phase 3: Inventory Management---

# Phases 3-6: Remaining Work## Phase 3: Inventory Management*To be developed*## Phase 4: Operational Records*To be developed*## Phase 5: Compliance & Utilities*To be developed*## Phase 6: Final Validation & Go-Live*To be developed*